# 3 — Explore the Icechunk store interactively

Widgets over the live store, so any region and period is available rather than the baked subset
in the website. Run with `panel serve 03_explore_live.ipynb --show`, or just execute the cells —
the dashboard renders inline in Jupyter.

The AOI subset is materialised on load, which is when virtual chunks are pulled from GCS. Keep the
box and the period modest.

In [ ]:
import numpy as np, pandas as pd, xarray as xr
import panel as pn, holoviews as hv, hvplot.xarray  # noqa: F401

pn.extension(sizing_mode="stretch_width")
hv.extension("bokeh")

## Connect

`apply_packing` is what decodes a store built by notebook 1 — without it the variables come back as raw int16.

In [ ]:
# --- reader: same three functions as notebook 1, inlined so this notebook stands alone ---
import os
from pathlib import Path
import icechunk

dotenv_path = Path.home() / "dotenv" / "protocoast.env"
if dotenv_path.exists():
    from dotenv import load_dotenv
    load_dotenv(dotenv_path, override=True)

BUCKET_HOST = "https://gcp-public-data-arco-era5.storage.googleapis.com"
CHUNK_ROOT  = BUCKET_HOST + "/"
BUCKET, STORE_NAME = "protocoast-data", "era5-sl-icechunk-v1"
SF_SUFFIX, AO_SUFFIX = "_scale_factor", "_add_offset"

storage = icechunk.s3_storage(bucket=BUCKET, prefix=f"icechunk/{STORE_NAME}", from_env=True,
                              endpoint_url=os.environ.get("ENDPOINT_URL"),
                              region="not-used", force_path_style=True)
repo_config = icechunk.RepositoryConfig.default()
repo_config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(url_prefix=CHUNK_ROOT, store=icechunk.http_store()))
credentials = icechunk.containers_credentials({CHUNK_ROOT: None})

def apply_packing(dset):
    out = dset
    for short in list(out.data_vars):
        sf, ao = short + SF_SUFFIX, short + AO_SUFFIX
        if sf in out.coords and ao in out.coords:
            attrs = dset[short].attrs
            out = out.assign({short: out[short] * out[sf] + out[ao]})
            out[short].attrs = attrs
    return out.drop_vars([c for c in out.coords
                          if c.endswith((SF_SUFFIX, AO_SUFFIX))], errors="ignore")

def open_era5(branch="main"):
    repo = icechunk.Repository.open(storage, repo_config,
                                    authorize_virtual_chunk_access=credentials)
    return apply_packing(xr.open_zarr(repo.readonly_session(branch).store,
                                      consolidated=False, chunks={}))

ds = open_era5()
print(ds.sizes, "|", str(ds.time.values[0])[:16], "->", str(ds.time.values[-1])[:16])

In [ ]:
VAR_LABEL = {"tp": "Total precipitation", "t2m": "2 m temperature"}
UNITS     = {"tp": "mm", "t2m": "degC"}
CMAP      = {"tp": "BuPu", "t2m": "RdYlBu_r"}
FREQ      = {"Hourly": "1h", "3-hourly": "3h", "6-hourly": "6h", "Daily": "1D",
             "Weekly": "7D", "Monthly": "MS", "Seasonal": "QS-DEC", "Annual": "YS"}

var    = pn.widgets.Select(name="Variable", options={v: k for k, v in VAR_LABEL.items()})
res    = pn.widgets.Select(name="Temporal resolution", options=list(FREQ), value="Daily")
agg    = pn.widgets.Select(name="Statistic", options=["sum", "mean", "min", "max"], value="sum")
fac    = pn.widgets.Select(name="Spatial resolution",
                           options={"0.25 deg (native)": 1, "0.50 deg": 2,
                                    "1.0 deg": 4, "2.0 deg": 8}, value=1)
method = pn.widgets.RadioButtonGroup(options={"Area-weighted": "mean", "Subsample": "subsample"},
                                     value="mean")
bbox   = pn.widgets.Select(name="Area of interest", options={
    "Po basin":      (7.0, 44.0, 12.5, 46.75),
    "Liguria":       (7.0, 43.75, 9.5, 45.25),
    "Alps":          (6.0, 45.5, 13.5, 47.75),
    "Tiber basin":   (11.0, 41.25, 13.5, 43.5),
    "Sicily":        (12.25, 36.5, 15.75, 38.5),
})
t0 = pn.widgets.DatePicker(name="Start", value=pd.Timestamp(ds.time.values[0]).date())
t1 = pn.widgets.DatePicker(name="End",   value=pd.Timestamp(ds.time.values[-1]).date())
go = pn.widgets.Button(name="Load & plot", button_type="success")

In [ ]:
def area_weighted_mean(da):
    return da.weighted(np.cos(np.deg2rad(da["latitude"]))).mean(("latitude", "longitude"))

def coarsen_space(da, factor, method="mean"):
    if factor == 1:
        return da
    if method == "subsample":
        return da.isel(latitude=slice(None, None, factor), longitude=slice(None, None, factor))
    w   = np.cos(np.deg2rad(da["latitude"]))
    num = (da * w).coarsen(latitude=factor, longitude=factor, boundary="trim").sum()
    den = w.coarsen(latitude=factor, boundary="trim").sum()
    return (num / den).assign_attrs(da.attrs)

def cell_area_km2(latitude, dlat=0.25, dlon=0.25):
    r = np.deg2rad(1.0)
    return (6371.0088 ** 2) * (dlon * r) * np.abs(
        np.sin(np.deg2rad(latitude + dlat / 2)) - np.sin(np.deg2rad(latitude - dlat / 2)))

def load():
    w, s, e, n = bbox.value
    da = ds[var.value].sel(longitude=slice(w, e), latitude=slice(n, s),
                           time=slice(str(t0.value), str(t1.value) + "T23:59")).load()
    da = da - 273.15 if var.value == "t2m" else da * 1000.0
    da.attrs["units"] = UNITS[var.value]
    field  = coarsen_space(getattr(da.resample(time=FREQ[res.value]), agg.value)(),
                           fac.value, method.value)
    series = getattr(area_weighted_mean(da).resample(time=FREQ[res.value]), agg.value)()
    area   = float(cell_area_km2(da.latitude.values).sum() * da.sizes["longitude"])
    return da, field, series, area

In [ ]:
@pn.depends(go.param.clicks)
def dashboard(_):
    if not go.clicks:
        return pn.pane.Markdown("Choose an area and period, then **Load & plot**.")
    da, field, series, area = load()
    ti = pn.widgets.IntSlider(name="Period", start=0, end=field.sizes["time"] - 1, value=0)

    @pn.depends(ti.param.value)
    def mapview(i):
        f = field.isel(time=i)
        return f.hvplot.quadmesh(x="longitude", y="latitude", cmap=CMAP[var.value],
                                 clabel=f"{VAR_LABEL[var.value]} [{UNITS[var.value]}]",
                                 frame_width=600, frame_height=420,
                                 title=f"{pd.Timestamp(f.time.values):%Y-%m-%d %H:%M} - "
                                       f"{0.25*fac.value:.2f} deg")

    ts = series.hvplot(x="time", kind="bar" if var.value == "tp" and series.sizes["time"] <= 80
                       else "line", frame_width=600, frame_height=230,
                       ylabel=f"AOI mean [{UNITS[var.value]}]")
    if var.value == "tp":
        ts = (ts + series.cumsum().hvplot(x="time", frame_width=600, frame_height=190,
                                          ylabel="cumulative [mm]")).cols(1)
    vol = (float(series.mean()) / 1000 * area * 1e6 / 1e9) if var.value == "tp" else float("nan")
    return pn.Column(
        pn.pane.Markdown(f"**{da.sizes['time']}** hourly steps - AOI **{area:,.0f} km2** - "
                         f"**{field.sizes['time']}** periods"
                         + (f" - mean water volume per period **{vol:.3f} km3**"
                            if var.value == "tp" else "")),
        ti, pn.Tabs(("Map", mapview), ("Series", ts)))

pn.Row(pn.Column(var, res, agg, fac, method, bbox, t0, t1, go, width=300),
       dashboard).servable()